In [19]:
from config import Bundle
import duckdb
import os
import polars as pl
import json
from IPython.display import display

In [20]:
def read_latest_raw_api_files(b1: Bundle) -> pl.DataFrame:
    """
    Read the latest raw API files from the bronze database.
    """
    query = f"""
        WITH latest_run AS (
            SELECT
                run_id,
                project_type
            FROM (
                SELECT
                    run_id,
                    project_type,
                    MAX(c_pull_timestamp_utc) AS max_pull_timestamp,
                    ROW_NUMBER() OVER (
                        PARTITION BY project_type
                        ORDER BY
                            MAX(c_pull_timestamp_utc) DESC,
                            run_id DESC
                    ) AS rn
                FROM {b1.raw_api_file_tables_table_name}
                GROUP BY
                    run_id,
                    project_type
            )
            WHERE rn = 1
        )

        SELECT
            r.run_id,
            r.project_type,
            r.project_id,
            r.payload,
            r.c_pull_timestamp_utc
        FROM {b1.raw_api_file_tables_table_name} r

        INNER JOIN latest_run l
            ON r.run_id = l.run_id
            AND r.project_type = l.project_type
    """

    with duckdb.connect(
        b1.bronze_db_path,
        read_only=True,
    ) as bronze_con:
        bronze_con.execute("SET arrow_large_buffer_size = true")
        return bronze_con.execute(query).pl()

In [21]:
def get_raw_api_file_counts(
    b1: Bundle,
) -> pl.DataFrame:
    """
    Return project payload and version counts for the latest
    raw API file ingestion.

    One Bronze row represents one project.
    Each payload contains an array of all versions for that project.
    """

    query = f"""
        WITH latest_run AS (
            SELECT
                project_type,
                run_id
            FROM (
                SELECT
                    project_type,
                    run_id,
                    MAX(c_pull_timestamp_utc) AS max_pull_timestamp,
                    ROW_NUMBER() OVER (
                        PARTITION BY project_type
                        ORDER BY MAX(c_pull_timestamp_utc) DESC
                    ) AS row_num
                FROM {b1.raw_api_file_tables_table_name}
                GROUP BY
                    project_type,
                    run_id
            )
            WHERE row_num = 1
        )

        SELECT
            r.run_id,
            r.project_type,

            COUNT(*)::BIGINT
                AS project_payloads,

            SUM(
                json_array_length(r.payload)
            )::BIGINT
                AS versions_fetched

        FROM {b1.raw_api_file_tables_table_name} r

        INNER JOIN latest_run l
            ON r.run_id = l.run_id
            AND r.project_type = l.project_type

        GROUP BY
            r.run_id,
            r.project_type

        ORDER BY
            r.project_type
    """

    with duckdb.connect(
        b1.bronze_db_path,
        read_only=True,
    ) as bronze_con:

        return bronze_con.execute(query).pl()

In [26]:
def normalize_version_value(value):
    """
    Normalize API values before Polars schema inference.

    Rules:
    - dictionaries are preserved as JSON strings
    - nested lists containing dictionaries/lists are preserved as JSON strings
    - empty lists become None so Polars does not infer List(Null)
    - simple populated lists remain native Polars list columns
    - scalar values are unchanged
    """

    if isinstance(value, dict):
        return json.dumps(value)

    if isinstance(value, list):

        # An empty list has no inferable inner dtype.
        # Convert it to null so populated rows determine the List dtype.
        if not value:
            return None

        # Complex nested arrays such as dependencies/files are kept
        # intact as JSON rather than inferred as nested Struct columns.
        contains_nested_values = any(
            isinstance(item, (dict, list))
            for item in value
        )

        if contains_nested_values:
            return json.dumps(value)

        # Simple arrays such as loaders/game_versions can remain
        # native Polars List columns.
        return value

    return value


def unpack_api_file_tables(
    raw_df: pl.DataFrame,
) -> pl.DataFrame:
    """
    Expand each project's version payload into one row per Modrinth version.

    Bronze grain:
        one row per run_id + project_type + project_id

    Silver grain:
        one row per Modrinth version

    Bronze metadata remains authoritative when the API payload contains
    a field with the same name.
    """
    version_rows = []

    metadata_columns = [
        column
        for column in raw_df.columns
        if column != "payload"
    ]

    for row in raw_df.iter_rows(named=True):

        # Metadata comes from the Bronze envelope.
        metadata = {
            column: row[column]
            for column in metadata_columns
        }

        versions = json.loads(row["payload"])

        if not versions:
            continue

        for version in versions:

            version_data = {
                key: normalize_version_value(value)
                for key, value in version.items()
            }

            # Remove payload fields that duplicate authoritative metadata.
            for metadata_column in metadata_columns:
                version_data.pop(
                    metadata_column,
                    None,
                )

            # Metadata is written last and therefore remains authoritative.
            version_rows.append({
                **version_data,
                **metadata,
            })

    if not version_rows:
        return pl.DataFrame()

    return pl.from_dicts(
        version_rows,
        infer_schema_length=None,
        strict=False,
    )

In [23]:
def write_polars_to_silver(
    b1: Bundle,
    df: pl.DataFrame,
    table_name: str,
) -> None:
    with duckdb.connect(b1.silver_db_path) as silver_con:
        silver_con.register("source_df", df)

        silver_con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM source_df
        """)

        silver_con.unregister("source_df")

In [27]:
def create_base_api_file_tables(b1: Bundle) -> None:

    # Read the latest Bronze version payloads.
    # One row = one project's JSON array of versions.
    raw_df = read_latest_raw_api_files(b1)

    print(
        f"Read in {raw_df.height:,} project version payloads from bronze."
    )

    # Display counts of projects and actual versions.
    raw_count_df = get_raw_api_file_counts(b1)

    display(raw_count_df)

    # Explode each project's JSON array into individual version rows.
    silver_df = unpack_api_file_tables(raw_df)

    print(
        f"Expanded into {silver_df.height:,} version rows."
    )

    # Write flattened version data to Silver.
    write_polars_to_silver(
        b1=b1,
        df=silver_df,
        table_name=b1.base_api_file_tables_table_name,
    )

In [28]:
#initialize object
b1 = Bundle()

#create silver layer with databases and tables initialized
b1.build_layer_directory(b1.silver_env_folder_path)
b1.init_db('silver')

#bring clean silver data and enforce schema
create_base_api_file_tables(b1)

Read in 157,730 project version payloads from bronze.


run_id,project_type,project_payloads,versions_fetched
str,str,i64,i64
"""9686352e-8206-416a-a615-74f4da…","""datapack""",14164,94120
"""9686352e-8206-416a-a615-74f4da…","""minecraft_java_server""",2016,3632
"""9686352e-8206-416a-a615-74f4da…","""mod""",72787,952384
"""9686352e-8206-416a-a615-74f4da…","""modpack""",18115,167093
"""9686352e-8206-416a-a615-74f4da…","""plugin""",16847,113858
"""9686352e-8206-416a-a615-74f4da…","""resourcepack""",32967,130865
"""9686352e-8206-416a-a615-74f4da…","""shader""",834,3767


Expanded into 1,465,719 version rows.


In [29]:
with duckdb.connect(b1.silver_db_path) as silver_con:
    silver_df = silver_con.execute(f"""
        SELECT *
        FROM {b1.base_api_file_tables_table_name}
    """).pl()
    
with pl.Config(
        tbl_rows = 5,
        tbl_cols = 35):

            display(silver_df)

game_versions,loaders,environment,id,author_id,featured,name,version_number,changelog_url,date_published,downloads,version_type,status,requested_status,files,dependencies,run_id,project_type,project_id,c_pull_timestamp_utc
list[str],list[str],str,str,str,bool,str,str,i32,str,i64,str,str,i32,str,str,str,str,str,"datetime[μs, America/Los_Angeles]"
"[""1.21.11""]","[""paper"", ""purpur""]","""unknown""","""45I6xzlS""","""40kDmbIK""",false,"""Vitamin+""","""2.0.1""",null,"""2026-03-07T14:47:44.452290Z""",180,"""release""","""listed""",null,"""[{""id"": ""Swja1K05"", ""hashes"": …",null,"""9686352e-8206-416a-a615-74f4da…","""plugin""","""wKw0THQX""",2026-08-22 09:47:10.695039 PDT
"[""1.21.11""]","[""paper"", ""purpur""]","""unknown""","""YL9qgFJp""","""40kDmbIK""",false,"""Vitamin+""","""2.0""",null,"""2026-03-04T19:19:37.296279Z""",29,"""release""","""listed""",null,"""[{""id"": ""1u0FeGV3"", ""hashes"": …",null,"""9686352e-8206-416a-a615-74f4da…","""plugin""","""wKw0THQX""",2026-08-22 09:47:10.695039 PDT
"[""1.21"", ""1.21.1"", … ""1.21.8""]","[""paper"", ""purpur"", ""spigot""]","""unknown""","""6LwDGgLx""","""40kDmbIK""",false,"""Vitamin+""","""1.7.1""",null,"""2025-07-19T22:38:15.197193Z""",332,"""release""","""listed""",null,"""[{""id"": ""tGMhQovA"", ""hashes"": …",null,"""9686352e-8206-416a-a615-74f4da…","""plugin""","""wKw0THQX""",2026-08-22 09:47:10.695039 PDT
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"[""1.19"", ""1.19.1"", … ""1.19.4""]","[""paper"", ""purpur"", ""spigot""]","""unknown""","""2TzLXKCx""","""WDStPNZD""",false,"""ExplosionProtector 1.0""","""1.0""",null,"""2025-04-27T17:22:42.123454Z""",30,"""release""","""listed""",null,"""[{""id"": ""B9oDNXDs"", ""hashes"": …","""[{""version_id"": null, ""project…","""9686352e-8206-416a-a615-74f4da…","""plugin""","""wKqja9iv""",2026-08-22 09:47:10.465220 PDT
"[""1.21"", ""1.21.1"", … ""1.21.5""]","[""paper"", ""purpur"", ""spigot""]","""unknown""","""9r1r4F21""","""WDStPNZD""",false,"""ExplosionProtector 1.0""","""1.0""",null,"""2025-04-07T01:49:22.017118Z""",49,"""release""","""listed""",null,"""[{""id"": ""isFCsMvh"", ""hashes"": …","""[{""version_id"": null, ""project…","""9686352e-8206-416a-a615-74f4da…","""plugin""","""wKqja9iv""",2026-08-22 09:47:10.465220 PDT
